In [7]:
# A full business solution
## imports
import os
import requests
import json
from typing import List
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from IPython.display import Markdown,display,update_display
from openai import OpenAI

load_dotenv()
api_key = os.getenv('OPENAI_API_KEY')
if api_key and api_key[:8] == "sk-proj-":
    print("API Key looks good so far")
else:
    print("There might be a problem with your api key")

MODEL = 'gpt-4o-mini'
openai = OpenAI()


API Key looks good so far


In [8]:
class Website:

    """ A utility class to represent a website that we have scraped"""
    url: str
    title: str
    body : str
    links: List[str]
    text: str

    def __init__(self,url):
        self.url = url
        response = requests.get(url)
        self.body = response.content
        soup = BeautifulSoup(self.body,'html.parser')
        self.title = soup.title.text if soup.title else "No title exists"
        if soup.body:
            for irrelevant in soup.body(["script","style","img","input"]):
                irrelevant.decompose()
            self.text = soup.body.get_text(separator="\n",strip=True)
        else:
            self.text = " "
        links = [link.get('href') for link in soup.find_all('a')]
        self.links = [link for link in links if link]

    
    def get_contents(self):
        return f"Webpage title:\n{self.title}\n Webpage contents:\n {self.text}\n\n"
    


In [9]:
link_system_prompt = "You are provided with list of links found on a website.\
You are able to decide which of the links would be most relevant to include in a brochure about the company,\
such as links to an About page or a company page, or career/jobs pages.\n"

link_system_prompt += " you should responds in JSON format as in this examples:"

link_system_prompt += """ 
{
    "links": [
        {"type": "about page","url":"https://full.url/goes/here/about"},
        {"type": "careers page","url":"https://another.full.url/carrers"},
    ]
}
"""


def get_links_user_prompt(website):
    user_prompt = f"Here is the list of links for the website {website.url} -"
    user_prompt += "please decide which of these are relevant web links for a brochure about the company, \
    respond with the full https URL :https://full.url/goes/here/about \
    do not include terms of services,Privacy,email links.\n"
    user_prompt += "Links (some might be relative links):\n"
    user_prompt += "\n".join(website.links)
    return user_prompt


def get_links(url):
    website = Website(url)
    response = openai.chat.completions.create(
          model = MODEL,
          messages = [
                        {"role":"system","content": link_system_prompt},
                        {"role":"user","content": get_links_user_prompt(website)}
          ],
        response_format = {"type":"json_object"}
    )

    result = response.choices[0].message.content
    return json.loads(result)



#selected links
get_links("https://anthropic.com")
    
    

{'links': [{'type': 'company page',
   'url': 'https://www.anthropic.com/company'},
  {'type': 'careers page', 'url': 'https://www.anthropic.com/careers'},
  {'type': 'about page', 'url': 'https://www.anthropic.com/learn'},
  {'type': 'research page', 'url': 'https://www.anthropic.com/research'},
  {'type': 'events page', 'url': 'https://www.anthropic.com/events'},
  {'type': 'news page', 'url': 'https://www.anthropic.com/news'},
  {'type': 'constitution page',
   'url': 'https://www.anthropic.com/constitution'},
  {'type': 'economic futures page',
   'url': 'https://www.anthropic.com/economic-futures'},
  {'type': 'transparency page',
   'url': 'https://www.anthropic.com/transparency'},
  {'type': 'responsible scaling policy page',
   'url': 'https://www.anthropic.com/responsible-scaling-policy'}]}

In [10]:
#get all details

def get_all_details(url):
    result = "landing page:\n"
    result += Website(url).get_contents()
    links = get_links(url) # get selected url
    print("\nFound links",links)
    for link in links["links"]:
        result+= f"\n\n {link['type']}\n"
        result+= Website(link['url']).get_contents()
    return result
    
print(get_all_details("https://anthropic.com"))


Found links {'links': [{'type': 'about page', 'url': 'https://www.anthropic.com/company'}, {'type': 'careers page', 'url': 'https://www.anthropic.com/careers'}, {'type': 'events page', 'url': 'https://www.anthropic.com/events'}, {'type': 'news page', 'url': 'https://www.anthropic.com/news'}, {'type': 'research page', 'url': 'https://www.anthropic.com/research'}, {'type': 'economic futures page', 'url': 'https://www.anthropic.com/economic-futures'}, {'type': 'constitution page', 'url': 'https://www.anthropic.com/constitution'}, {'type': 'transparency page', 'url': 'https://www.anthropic.com/transparency'}, {'type': 'responsible scaling policy page', 'url': 'https://www.anthropic.com/responsible-scaling-policy'}]}
landing page:
Webpage title:
Home \ Anthropic
 Webpage contents:
 Skip to main content
Skip to footer
Research
Economic Futures
Commitments
Initiatives
Claude's Constitution
Transparency
Responsible Scaling Policy
Trust center
Security and compliance
Learn
Learn
Anthropic Acad

In [11]:
system_prompt = "You are an assistant that analyzes the contents of several relevants pages from a company website \
and creates a short brochure about the company for prospective customers,investors and recruits.Responds in Markdown.\
incude details of company culture,customers and careers/jobs if you have the information."
 
def get_brochure_user_prompt(company_name,url):
    user_prompt = f"You are looking at a company called: {company_name}\n"
    user_prompt += f"Here are the contents of it's landing page and other relevant pages; use this information to build a short brochure of the comapny in markdown.\n"
    user_prompt += get_all_details(url)
    user_prompt = user_prompt[:20_000] #truncate if more than 20000 characters
    return user_prompt



get_brochure_user_prompt("Anthropic","https://anthropic.com")



Found links {'links': [{'type': 'about page', 'url': 'https://www.anthropic.com/company'}, {'type': 'careers page', 'url': 'https://www.anthropic.com/careers'}]}


"You are looking at a company called: Anthropic\nHere are the contents of it's landing page and other relevant pages; use this information to build a short brochure of the comapny in markdown.\nlanding page:\nWebpage title:\nHome \\ Anthropic\n Webpage contents:\n Skip to main content\nSkip to footer\nResearch\nEconomic Futures\nCommitments\nInitiatives\nClaude's Constitution\nTransparency\nResponsible Scaling Policy\nTrust center\nSecurity and compliance\nLearn\nLearn\nAnthropic Academy\nTutorials\nUse cases\nEngineering at Anthropic\nDeveloper docs\nCompany\nAbout\nCareers\nEvents\nNews\nTry Claude\nTry Claude\nTry Claude\nLearn more about Claude\nProducts\nClaude\nClaude Code\nClaude Cowork\nClaude Platform\nPricing\nContact sales\nModels\nOpus\nSonnet\nHaiku\nLog in\nClaude.ai\nClaude Console\nEN\nThis is some text inside of a div block.\nLog in to Claude\nLog in to Claude\nLog in to Claude\nDownload app\nDownload app\nDownload app\nResearch\nEconomic Futures\nCommitments\nInitiati

In [13]:
def create_brochure(company_name,url):
    response = openai.chat.completions.create(
        model = MODEL,
        messages=[
            {"role":"system","content":system_prompt},
             {"role":"user","content":get_brochure_user_prompt("Anthropic","https://anthropic.com")}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))


create_brochure("Anthropic","https://anthropic.com")


Found links {'links': [{'type': 'company page', 'url': 'https://www.anthropic.com/company'}, {'type': 'careers page', 'url': 'https://www.anthropic.com/careers'}, {'type': 'research page', 'url': 'https://www.anthropic.com/research'}, {'type': 'news page', 'url': 'https://www.anthropic.com/news'}, {'type': 'events page', 'url': 'https://www.anthropic.com/events'}, {'type': 'economic futures page', 'url': 'https://www.anthropic.com/economic-futures'}, {'type': 'constitution page', 'url': 'https://www.anthropic.com/constitution'}, {'type': 'transparency page', 'url': 'https://www.anthropic.com/transparency'}, {'type': 'responsible scaling policy page', 'url': 'https://www.anthropic.com/responsible-scaling-policy'}]}


# Anthropic Company Brochure

## About Us
Anthropic is an AI safety and research company dedicated to building reliable, interpretable, and steerable AI systems. Founded with the vision that AI will profoundly impact the world, we aim to secure its benefits while mitigating its risks. As a Public Benefit Corporation, our mission is clear - to develop advanced AI technologies that serve humanity's long-term well-being.

## Our Products
We develop innovative AI products under the Claude brand:
- **Claude**: Our flagship AI designed to assist users in a helpful and trustworthy manner.
- **Claude Code**: Specialized tools for developers to enhance their coding experience.
- **Claude Cowork**: An AI platform designed to facilitate collaborative workflows.
- **Claude Platform**: A comprehensive suite of AI tools integrated into various business solutions.

## Customers
Anthropic serves a diverse range of industries including:
- **Healthcare**
- **Financial Services**
- **Education**
- **Government**
- **Nonprofits**

Our artificial intelligence solutions are designed to create positive impacts, enabling businesses and organizations to enhance their operations and better serve their communities.

## Career Opportunities
At Anthropic, we're always on the lookout for passionate and innovative individuals. We value diverse expertise and backgrounds. Whether you have a tech background or not, if you're drawn to tackling meaningful challenges, we want to hear from you.

### Employee Benefits
- Comprehensive health, dental, and vision insurance
- 22 weeks of paid parental leave
- Retirement plans with competitive matching
- Flexible paid time off
- Daily meals and snacks in the office

### Our Values
1. **Act for the global good**: Maximize positive outcomes for humanity.
2. **Hold light and shade**: Recognize both the benefits and risks of AI.
3. **Be good to our users**: Treat all users with kindness and respect.
4. **Ignite a race to the top on safety**: Lead the industry in AI safety standards.
5. **Do the simple thing that works**: Focus on impactful solutions.
6. **Be helpful, honest, and harmless**: Maintain a high-trust, low-ego environment.
7. **Put the mission first**: Our mission drives every decision we make.

## Company Culture
We foster an interdisciplinary collaboration among researchers, engineers, and policy experts, striving to learn and share insights on AI safety. Our workplace encourages kindness, transparency, and an empirical approach to problems to ensure that the development of AI technology is a robustly positive force for good.

### Join Us
Interested in making a mark in AI? Explore our open roles and become part of a team committed to shaping the future of artificial intelligence safely and ethically.

[Explore Open Roles](#)

---

### Contact Us
For more information about our products or career opportunities, please visit our website: [anthropic.com](https://www.anthropic.com)

---

*Together, we aim to build a future where AI serves as a force for good.*